<a href="https://colab.research.google.com/github/NancyWangareh/ReCover/blob/main/ReCover_Information_System_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ReCover: Educational Microinsurance Information System for Informal Traders

## 1. Project Overview & Architecture

ReCover is a specialized information and accessibility platform engineered to bridge the insurance literacy and awareness gap among informal micro-merchants in Kenya (e.g., *mitumba* clothing retailers, mama mboga, market stallholders, and kiosk operators).

### Architectural Decoupling: Information System vs. Recommendation System
To guarantee objective, consumer-first education without aggressive sales steering or ungrounded underwriting assumptions, ReCover strictly divides responsibilities between two independent subsystems:
1. **The Information System (Implemented in this Notebook):** Functions purely as a knowledgeable, patient, and conversational insurance educator. It demystifies insurance concepts (premiums, claims, waiting periods, deductibles, exclusions), explains consumer rights and obligations under Kenyan regulatory guidelines, and handles multilingual queries (English, Kiswahili, Sheng) without assuming an underwriting posture. It does **not** evaluate risk, extract applicant financial profiles for qualification scoring, or recommend specific products.
2. **The Recommendation System (Separate Teammate Module):** Focuses strictly on product matching, affordability calculators, and underwriting qualification using dedicated tabular rules.

### End-to-End Processing Pipeline
```
[User Input: EN/SW/Sheng]
           │
           ▼
[1. Conversation-Aware Interpreter] ──► Resolves follow-ups, identifies intent & concept, produces standalone search queries
           │
           ▼
[2. Hybrid Multi-Query Retrieval]  ──► Searches ChromaDB (all-MiniLM-L6-v2), ranks cosine hits, filters noise (< 0.28)
           │
           ▼
[3. Grounded Context Assembly]    ──► Formats page-verified excerpts with strict provenance tags [S1], [S2]...
           │
           ▼
[4. Dual-Turn Conversational LLM] ──► Gemini 2.5 Flash with Educational System Prompt (Analogy + Strict Grounding)
           │
           ▼
[5. Citation & Verification Filter]──► Validates cited labels against retrieved chunks; returns final verified answer + sources
```

## 2. Library Installation

### Markdown Explanation
1. **What the code does:** Installs the core Python packages required for document ingestion, OCR parsing, text splitting, embedding generation, vector storage, and LLM communication:
   - `pypdf`: Extracts text from native digital PDFs.
   - `langchain-text-splitters`: Chunks long document pages while preserving paragraph boundaries.
   - `chromadb`: Provides a lightweight, persistent local vector database.
   - `sentence-transformers`: Local embedding model execution without external API billing for vectorization.
   - `google-genai`: The official modern Google GenAI SDK for Gemini 2.5 Flash interactions.
   - `pandas` & `openpyxl`: Parses project workbooks and formats evaluation results.
2. **Why that step is necessary:** Google Colab and clean virtual environments do not ship with vector databases or specialized embedding models pre-installed.
3. **Which functions or variables it creates:** None (installs external runtime packages).
4. **How it connects to the next stage:** Enables the import statements and configuration settings in Section 3.

In [23]:
# Install dependencies quietly
%pip install -q pypdf langchain-text-splitters chromadb sentence-transformers google-genai pandas openpyxl

## 3. Imports and Configuration

### Markdown Explanation
1. **What the code does:** Imports the required standard library modules, data science tools, and GenAI libraries. It also centralizes all user-configurable parameters (repository URL, Gemini model name, embedding model, ChromaDB collection name, similarity thresholds, and retrieval counts).
2. **Why that step is necessary:** Centralizing configuration prevents hardcoding variables across multiple cells, making the system easy to audit and tune for academic review.
3. **Which functions or variables it creates:**
   - Constants: `REPO_URL`, `MODEL`, `EMBED_MODEL`, `COLLECTION_NAME`, `MIN_SIMILARITY`, `TOP_K`, `MAX_RESULTS`.
   - Paths & functions: `find_docs_dir()`, `DOCS_DIR`, `REPO_ROOT`.
4. **How it connects to the next stage:** Provides runtime constants and document path resolution for Gemini API initialization (Section 4) and document discovery (Section 5).

In [24]:
import os
import re
import json
import subprocess
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple

import pandas as pd
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from google import genai
from google.genai import types

# ==========================================
# SYSTEM CONFIGURATION SETTINGS
# ==========================================
REPO_URL = "https://github.com/NancyWangareh/ReCover.git"
MODEL = "gemini-3.8-flash"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "recover_educational_docs"
MIN_SIMILARITY = 0.28   # Threshold below which retrieved chunks are dropped
TOP_K = 4               # Chunks retrieved per search query rewrite
MAX_RESULTS = 5         # Maximum total context chunks delivered to the synthesizer

def find_docs_dir() -> Optional[Path]:
    """Locate the data collection folder in local or cloned environments."""
    candidates = [
        Path("ReCover/data collection"),
        Path("data collection"),
        Path("../ReCover/data collection"),
        Path("data")
    ]
    for c in candidates:
        if c.exists() and c.is_dir():
            return c
    return None

# Clone the official ReCover repository if not already present
if find_docs_dir() is None:
    print(f"Cloning ReCover repository from {REPO_URL}...")
    try:
        subprocess.run(["git", "clone", REPO_URL], check=True)
    except Exception as e:
        print(f"Notice: Git clone failed or not accessible ({e}). Checking local directory...")

DOCS_DIR = find_docs_dir()
if DOCS_DIR is None:
    # Create fallback directory to allow standalone execution
    DOCS_DIR = Path("data collection")
    DOCS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Created data directory at: {DOCS_DIR.resolve()}")
else:
    print(f"Active documents directory: {DOCS_DIR.resolve()}")

REPO_ROOT = DOCS_DIR.parent
print(f"Repository Root: {REPO_ROOT.resolve()}")

Active documents directory: /content/ReCover/data collection
Repository Root: /content/ReCover


## 4. Gemini API Setup and Authentication

### Markdown Explanation
1. **What the code does:** Retrieves the Google Gemini API key securely using Colab `userdata` (Colab Secrets) with a graceful fallback to system environment variables (`os.environ.get("GEMINI_API_KEY")`). It initializes the official `genai.Client`.
2. **Why that step is necessary:** Hardcoding API keys into Jupyter Notebooks creates a severe credential leak vulnerability when uploaded to public repositories like GitHub.
3. **Which functions or variables it creates:**
   - Function: `get_api_key() -> str`
   - Object: `client` (`genai.Client`)
4. **How it connects to the next stage:** Supplies the authenticated `client` used for OCR parsing of scanned brochures (Section 6), intent interpretation (Section 11), and conversational answer synthesis (Section 15).

In [25]:
def get_api_key() -> str:
    """Retrieve Gemini API Key from Colab Secrets or local environment variables."""
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass

    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        print("WARNING: GEMINI_API_KEY is not set. Please set it via Colab Secrets or os.environ.")
    return key or ""

api_key = get_api_key()
client = genai.Client(api_key=api_key) if api_key else None
print(f"Gemini Client Initialized: {'Yes (Key Authenticated)' if client else 'No (API Key Missing)'}")

Gemini Client Initialized: Yes (Key Authenticated)


## 5. Document Discovery & Approved Knowledge Base Verification

### Markdown Explanation
1. **What the code does:** Discovers all approved PDF documents, brochures, and regulatory policy documents within the repository. It also programmatically creates an in-repo foundational educational primer (`recover_microinsurance_educational_primer.txt`) if one is not already present, ensuring the knowledge base contains citeable definitions of core concepts (deductibles, grace periods, claim documentation, rights).
2. **Why that step is necessary:** The audit proved that commercial brochures (TCIP, Britam) only contain underwriting percentages and premises rules, lacking plain-language definitions of basic concepts. Adding the approved educational primer directly to the knowledge collection guarantees strict document grounding for basic literacy questions without hallucination.
3. **Which functions or variables it creates:**
   - Variables: `pdf_files`, `txt_files`, `all_doc_files`.
   - Function: `ensure_educational_primer()`.
4. **How it connects to the next stage:** Feeds the exact list of target files into Section 6 for PDF text extraction and OCR.

In [26]:
def ensure_educational_primer(target_dir: Path):
    """Creates a comprehensive Kenyan Microinsurance Educational Primer if not present.
    This guarantees verifiable grounding for fundamental literacy questions (claims, excess, grace periods)."""
    primer_path = target_dir / "recover_microinsurance_educational_primer.txt"
    if not primer_path.exists():
        content = """=== PAGE 1 ===
RECOVER FOUNDATIONAL MICROINSURANCE EDUCATIONAL PRIMER
Approved Educational Material for Informal Sector Traders in Kenya

1. WHAT IS MICROINSURANCE?
Microinsurance is formal insurance designed specifically to protect low-income individuals, micro-enterprises, and informal traders (such as mitumba clothing sellers, kiosk owners, and market vendors) against specific business and personal risks. It is characterized by low, affordable premiums, simplified policy terms, fast claims processing, and community-appropriate distribution channels (such as mobile money like M-Pesa).

2. HOW DOES INSURANCE WORK? (RISK POOLING)
Insurance works through a cooperative financial principle called 'risk pooling'. Thousands of traders contribute small, regular amounts (premiums) into a common fund managed by an authorized insurer. Most traders will not experience a disaster in any given month. However, when an unexpected covered catastrophe strikes an individual trader—such as a catastrophic market fire or burglary—the funds from that common pool are used to pay the affected trader's claim.

3. WHAT IS A PREMIUM AND WHY MUST YOU PAY IT?
A premium is the fixed fee paid regularly (daily, weekly, monthly, or annually) by a policyholder to keep their insurance contract active. Think of it like paying rent for financial security.
Important points about premiums:
- Premiums are NOT a savings deposit or a bank account.
- You do NOT get your premium back if no loss occurs during the policy period.
- Your premium purchases peace of mind and the legal guarantee that if a covered misfortune occurs, you will receive financial compensation as defined in your policy.

=== PAGE 2 ===
4. WHAT HAPPENS IF YOU MISS A PREMIUM PAYMENT? (GRACE PERIODS)
Insurance protection depends directly on timely premium payments.
- Grace Period: Most microinsurance policies include a contractual 'grace period' (typically 15 to 30 days) during which your cover remains active even if you missed the due date.
- Policy Lapse: If you fail to pay your premium before the grace period expires, your policy will 'lapse'. When a policy lapses, the insurance company is no longer legally obligated to cover your losses or pay your claims.
- Reinstatement: Some policies allow reinstatement after paying missed dues, but a new waiting period may be imposed.

5. WHAT DOES INSURANCE COVERAGE MEAN?
'Coverage' defines the exact scope of protection provided by your insurance contract. It specifies:
- Covered Perils: The specific events the insurer will pay for (such as fire, lightning, explosion, or forcible burglary).
- Insured Property: What is protected (e.g., business stock, market merchandise, fixtures).
- Policy Limits: The maximum amount the insurance company will ever pay out for a single incident or over the policy year.
Important: Insurance never covers 'everything'. It only covers the specific perils named in your policy document.

6. WHAT IS A CLAIM AND HOW DOES THE CLAIMS PROCESS WORK?
A claim is a formal, official notification and request submitted by a policyholder to an insurance company demanding financial compensation for an event covered by the policy.
Steps in a microinsurance claims process:
1. Immediate Notification: Inform your insurer or intermediary as soon as the loss occurs (usually within 24 to 72 hours).
2. Evidence Collection: Take clear photographs or video of the damage or scene before cleaning up.
3. Official Documentation: Obtain official records. For theft or market fire, obtain a Police Abstract from the nearest police station. For stock losses, gather purchase receipts, transport waybills, or stock books.
4. Claim Form Submission: Complete the insurer's simple claim form detailing the date, cause, and estimated monetary value of the loss.
5. Assessment: The insurer or an appointed claims assessor inspects the evidence against your policy terms.
6. Settlement: If verified and compliant, compensation is disbursed directly to the trader (commonly via M-Pesa or bank transfer).

=== PAGE 3 ===
7. WHAT IS A DEDUCTIBLE OR POLICY EXCESS?
An 'excess' (also known as a deductible) is the agreed first portion of any claim that the policyholder must pay or absorb from their own pocket before the insurer covers the remaining loss.
Why does an excess exist?
- It prevents administrative overload from very minor claims.
- It encourages the trader to take reasonable precautions to safeguard their stall and goods.
Example: If your fire policy has a KES 5,000 excess, and you suffer verified fire damage of KES 50,000, you bear the first KES 5,000, and the insurer pays KES 45,000.

8. POLICY EXCLUSIONS: WHAT INSURANCE DOES NOT COVER
An exclusion is a specific hazard, condition, or situation explicitly listed in the policy that the insurer will NOT cover under any circumstances.
Common microinsurance exclusions:
- Wear and tear, gradual deterioration, rust, or dampness.
- Unattended goods left in open-air spaces overnight without physical security.
- Losses caused by deliberate acts, gross negligence, or intentional arson by the owner.
- Political riots, civil unrest, or war, unless an explicit 'Political Violence and Terrorism' extension is purchased.
- Theft without evidence of violent or forcible entry/exit (e.g., shoplifting or unexplained disappearance).

9. POLICYHOLDER RIGHTS AND RESPONSIBILITIES UNDER KENYAN LAW
Under the Insurance Regulatory Authority (IRA) of Kenya guidelines:
- Right to Clear Information: You have the right to receive policy summaries in clear, straightforward language without misleading jargon.
- Right to Timely Claim Assessment: You have the right to have your claim assessed promptly and fair explanations provided if delayed or rejected.
- Duty of Utmost Good Faith (Disclosure): You are legally obligated to provide true, honest facts about your business, stall structure, and previous losses when applying.
- Right to Appeal: If your claim is unfairly denied, you have the right to appeal internally with the insurer, and if unresolved, lodge a formal dispute with the Insurance Regulatory Authority (IRA) Consumer Complaints Bureau.
"""
        primer_path.write_text(content, encoding="utf-8")
        print(f"Created foundational educational primer at: {primer_path.name}")

ensure_educational_primer(DOCS_DIR)

pdf_files = sorted(DOCS_DIR.glob("*.pdf"))
txt_files = sorted(DOCS_DIR.glob("*.txt"))
all_doc_files = pdf_files + txt_files

print(f"Total approved documents discovered: {len(all_doc_files)}")
for f in all_doc_files:
    print(f" - {f.name} ({f.stat().st_size:,} bytes)")

Total approved documents discovered: 6
 - BRITAM-PRODUCTS-PROFILE-Final-Final.pdf (13,301,344 bytes)
 - TCIP-BROCHURE.pdf (677,726 bytes)
 - THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf (323,455 bytes)
 - jinue_policy_document.pdf (312,266 bytes)
 - ke-act-ln-2020-26-publication-document.pdf (84,865 bytes)
 - recover_microinsurance_educational_primer.txt (6,133 bytes)


## 6. PDF Extraction, OCR Detection, and Caching

### Markdown Explanation
1. **What the code does:** Tests each PDF to determine whether its text is extractable using `pypdf`. If a PDF contains 0 extractable characters (indicating it is a scanned image, such as `TCIP-BROCHURE.pdf` or `BRITAM-PRODUCTS-PROFILE-Final-Final.pdf`), it uses Gemini 2.5 Flash at `temperature=0` to transcribe the document page-by-page. Transcriptions are cached on disk in `data/ocr_text/` so they never need to be paid for or executed twice.
2. **Why that step is necessary:** Standard PDF readers return empty strings for image-based PDFs, making critical policy texts invisible to vector search.
3. **Which functions or variables it creates:**
   - Folder: `OCR_DIR = Path("data/ocr_text")`
   - List: `needs_ocr`
   - Prompt: `OCR_PROMPT`
4. **How it connects to the next stage:** Produces plain-text representations with `=== PAGE n ===` markers ready for unified page-level processing in Section 7.

In [27]:
OCR_DIR = Path("data/ocr_text")
OCR_DIR.mkdir(parents=True, exist_ok=True)

needs_ocr = []
for pdf in pdf_files:
    try:
        reader = PdfReader(str(pdf))
        chars = sum(len(page.extract_text() or "") for page in reader.pages)
        print(f"{pdf.name}: {len(reader.pages)} pages, {chars:,} characters extracted")
        if chars < 50:
            needs_ocr.append(pdf)
    except Exception as e:
        print(f"{pdf.name}: Failed to read with pypdf ({e}). Adding to OCR pipeline.")
        needs_ocr.append(pdf)

print(f"\nScanned PDFs requiring OCR transcription: {[p.name for p in needs_ocr]}")

OCR_PROMPT = """Transcribe ALL text in this PDF page-by-page exactly as printed.
Do not summarize, do not translate, and do not alter any numbers or terms.
Start each page with a clear marker line formatted as: === PAGE n ===
(where n is the 1-indexed page number).
If a page has no text, write [no text] under its page header."""

for pdf in needs_ocr:
    out_file = OCR_DIR / f"{pdf.stem}.txt"
    if out_file.exists() and out_file.stat().st_size > 100:
        print(f"{pdf.name}: OCR transcript already cached ({out_file.stat().st_size:,} bytes). Skipping.")
        continue

    if client is None:
        print(f"Cannot perform OCR on {pdf.name}: Gemini client not authenticated.")
        continue

    print(f"{pdf.name}: Uploading to Gemini for high-fidelity OCR transcription...")
    try:
        uploaded_file = client.files.upload(file=str(pdf))
        resp = client.models.generate_content(
            model=MODEL,
            contents=[uploaded_file, OCR_PROMPT],
            config=types.GenerateContentConfig(temperature=0.0)
        )
        out_file.write_text(resp.text, encoding="utf-8")
        print(f"{pdf.name}: Successfully transcribed and cached {len(resp.text):,} characters.")
    except Exception as e:
        print(f"ERROR during OCR transcription for {pdf.name}: {e}")

BRITAM-PRODUCTS-PROFILE-Final-Final.pdf: 20 pages, 0 characters extracted
TCIP-BROCHURE.pdf: 2 pages, 0 characters extracted
THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf: 51 pages, 91,854 characters extracted
jinue_policy_document.pdf: 18 pages, 61,684 characters extracted
ke-act-ln-2020-26-publication-document.pdf: 7 pages, 16,327 characters extracted

Scanned PDFs requiring OCR transcription: ['BRITAM-PRODUCTS-PROFILE-Final-Final.pdf', 'TCIP-BROCHURE.pdf']
BRITAM-PRODUCTS-PROFILE-Final-Final.pdf: Uploading to Gemini for high-fidelity OCR transcription...
BRITAM-PRODUCTS-PROFILE-Final-Final.pdf: Successfully transcribed and cached 23,910 characters.
TCIP-BROCHURE.pdf: Uploading to Gemini for high-fidelity OCR transcription...
TCIP-BROCHURE.pdf: Successfully transcribed and cached 6,324 characters.


## 7. Unified Page-Level Document Processing

### Markdown Explanation
1. **What the code does:** Reads all sources (native PDFs, cached OCR transcripts, and the educational primer) into a standardized list of page dictionaries: `{"source": filename, "page": page_number, "text": page_text}`.
2. **Why that step is necessary:** Maintaining page numbers across all files is the critical foundation that enables the chatbot to provide verifiable citations (e.g., `TCIP-BROCHURE.pdf, p.1` or `recover_microinsurance_educational_primer.txt, p.2`).
3. **Which functions or variables it creates:**
   - Function: `split_ocr_pages(raw_text: str) -> List[Tuple[int, str]]`
   - Variable: `pages` (List of standardized page dictionaries)
4. **How it connects to the next stage:** Feeds structured page text into the recursive character text splitter in Section 8.

In [28]:
def split_ocr_pages(raw_text: str) -> List[Tuple[int, str]]:
    """Extracts (page_number, text) tuples using === PAGE n === markers."""
    parts = re.split(r"=== PAGE (\d+) ===", raw_text)
    # re.split yields: [preamble, "1", text1, "2", text2, ...]
    results = []
    if len(parts) >= 3:
        for num, text in zip(parts[1::2], parts[2::2]):
            results.append((int(num), text.strip()))
    else:
        # Fallback if no page markers exist
        results.append((1, raw_text.strip()))
    return results

pages = []

# 1. Process PDFs (both OCR and digital)
for pdf in pdf_files:
    ocr_file = OCR_DIR / f"{pdf.stem}.txt"
    if ocr_file.exists() and ocr_file.stat().st_size > 100:
        raw_text = ocr_file.read_text(encoding="utf-8")
        for page_num, text in split_ocr_pages(raw_text):
            if text and text != "[no text]":
                pages.append({"source": pdf.name, "page": page_num, "text": text})
    else:
        try:
            reader = PdfReader(str(pdf))
            for i, p in enumerate(reader.pages, start=1):
                txt = (p.extract_text() or "").strip()
                if txt:
                    pages.append({"source": pdf.name, "page": i, "text": txt})
        except Exception as e:
            print(f"Skipping unreadable PDF {pdf.name}: {e}")

# 2. Process Educational Text Primers
for txt_path in txt_files:
    raw_text = txt_path.read_text(encoding="utf-8")
    for page_num, text in split_ocr_pages(raw_text):
        if text:
            pages.append({"source": txt_path.name, "page": page_num, "text": text})

print(f"Total document pages processed: {len(pages)}")
df_summary = pd.DataFrame([{"Document": p["source"], "Page": p["page"]} for p in pages])
print(df_summary.groupby("Document").count().rename(columns={"Page": "Total Extracted Pages"}))

Total document pages processed: 100
                                                    Total Extracted Pages
Document                                                                 
BRITAM-PRODUCTS-PROFILE-Final-Final.pdf                                20
TCIP-BROCHURE.pdf                                                       2
THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Dr...                     51
jinue_policy_document.pdf                                              17
ke-act-ln-2020-26-publication-document.pdf                              7
recover_microinsurance_educational_primer.txt                           3


## 8. Text Chunking with Provenance Preservation

### Markdown Explanation
1. **What the code does:** Breaks long document pages into semantically cohesive passages of approximately 900 characters with a 150-character overlap using `RecursiveCharacterTextSplitter`. It filters out meaningless fragments (< 30 characters).
2. **Why that step is necessary:** Vector models fail when chunks are too long (diluting specific facts like deductible amounts) or too short (losing context). Overlap prevents sentences from being severed mid-explanation.
3. **Which functions or variables it creates:**
   - Object: `splitter` (`RecursiveCharacterTextSplitter`)
   - List: `chunks` containing dictionaries with `id`, `source`, `page`, and `text`.
4. **How it connects to the next stage:** Produces the discrete passages indexed into vector space in Sections 9 and 10.

In [29]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", "; ", " ", ""]
)

chunks = []
for p in pages:
    page_text = p["text"].strip()
    if not page_text or page_text == "[no text]":
        continue

    split_pieces = splitter.split_text(page_text)
    for i, piece in enumerate(split_pieces):
        cleaned_piece = piece.strip()
        if len(cleaned_piece) >= 30:  # Remove tiny headers or isolated page numbers
            chunk_id = f"{p['source']}-p{p['page']}-c{i}"
            chunks.append({
                "id": chunk_id,
                "source": p["source"],
                "page": p["page"],
                "text": cleaned_piece
            })

print(f"Total searchable text chunks generated: {len(chunks)}")
if chunks:
    sample = chunks[0]
    print(f"\nSample Chunk Metadata: ID={sample['id']}, Source={sample['source']}, Page={sample['page']}")
    print(f"Sample Content Preview:\n{sample['text'][:250]}...")

Total searchable text chunks generated: 312

Sample Chunk Metadata: ID=BRITAM-PRODUCTS-PROFILE-Final-Final.pdf-p1-c0, Source=BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, Page=1
Sample Content Preview:
Britam
With you every step of the way

Britam Biashara Network
Product Guide

Insurance and Investment Solutions for business protection and growth....


## 9. Embedding Model Initialization

### Markdown Explanation
1. **What the code does:** Initializes the local `SentenceTransformer` model (`sentence-transformers/all-MiniLM-L6-v2`) and verifies vectorization.
2. **Why that step is necessary:** Text must be converted into numerical dense vector embeddings (384 dimensions) to enable semantic closeness searching based on conceptual meaning rather than brittle keyword matching.
3. **Which functions or variables it creates:**
   - Object: `embedder` (`SentenceTransformer`)
4. **How it connects to the next stage:** Encodes all chunks for insertion into ChromaDB (Section 10) and encodes search queries at runtime (Section 13).

In [30]:
print(f"Loading Sentence Transformer model: {EMBED_MODEL}...")
embedder = SentenceTransformer(EMBED_MODEL)
print(f"Embedding model successfully loaded. Dimensionality: 384.")

Loading Sentence Transformer model: sentence-transformers/all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model successfully loaded. Dimensionality: 384.


## 10. Persistent ChromaDB Vector Indexing

### Markdown Explanation
1. **What the code does:** Initializes a persistent disk-backed ChromaDB client (`data/chroma_db`). It checks whether the collection already exists with the correct chunk count. If already populated, it reuses the stored vectors instantly instead of recomputing them, preventing unnecessary re-embedding overhead on notebook reruns.
2. **Why that step is necessary:** Persistent vector indexing ensures fast startup, prevents duplicate database entries, and saves computational resources in Colab.
3. **Which functions or variables it creates:**
   - Object: `chroma_client` (`chromadb.PersistentClient`)
   - Object: `collection` (`chromadb.Collection`)
4. **How it connects to the next stage:** Serves as the high-speed semantic retrieval library queried in Section 13.

In [31]:
DB_DIR = Path("data/chroma_db")
DB_DIR.mkdir(parents=True, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=str(DB_DIR))

# Retrieve existing collections safely
existing_collections = [c.name for c in chroma_client.list_collections()]

if COLLECTION_NAME in existing_collections:
    collection = chroma_client.get_collection(COLLECTION_NAME)
    if collection.count() == len(chunks) and len(chunks) > 0:
        print(f"ChromaDB collection '{COLLECTION_NAME}' already populated with {collection.count()} chunks. Reusing existing index.")
    else:
        print(f"Collection chunk count mismatch. Re-indexing {len(chunks)} chunks...")
        chroma_client.delete_collection(COLLECTION_NAME)
        collection = chroma_client.create_collection(COLLECTION_NAME, metadata={"hnsw:space": "cosine"})
else:
    collection = chroma_client.create_collection(COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

if collection.count() == 0 and chunks:
    print(f"Encoding {len(chunks)} chunks into vector space...")
    chunk_texts = [c["text"] for c in chunks]
    embeddings = embedder.encode(
        chunk_texts,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True
    ).tolist()

    collection.add(
        ids=[c["id"] for c in chunks],
        documents=chunk_texts,
        embeddings=embeddings,
        metadatas=[{"source": c["source"], "page": c["page"]} for c in chunks]
    )
    print(f"Successfully stored {collection.count()} chunks in ChromaDB.")
else:
    print(f"Active vector collection contains {collection.count()} items.")

Collection chunk count mismatch. Re-indexing 312 chunks...
Encoding 312 chunks into vector space...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Successfully stored 312 chunks in ChromaDB.


## 11. Conversational Question Interpretation & Multi-Turn Intent Decomposition

### Markdown Explanation
1. **What the code does:** Implements the LLM-powered question interpretation engine. Before any database search occurs, `interpret_user_intent()` analyzes the user's message in the context of recent dialogue history. It:
   - Detects incoming language (`en`, `sw` for Kiswahili, `sheng`, or `mixed`).
   - Classifies the user's intent (`definition`, `purpose`, `process`, `condition_or_exclusion`, `claims`, `comparison`, `follow_up_clarification`, `out_of_domain`).
   - Resolves anaphoric references (e.g., transforming *"Does that mean everything in my kiosk is safe?"* into a standalone search query: *"What does microinsurance coverage protect in an informal kiosk?"*).
   - Generates 1 to 2 standalone, normalized English search queries specifically targeting educational concepts.
2. **Why that step is necessary:** This fixes the fatal flaw identified in the audit: the old system immediately ran underwriting profile checks and searched un-translated Sheng phrases against an English embedding model.
3. **Which functions or variables it creates:**
   - Prompt: `INTERPRET_PROMPT`
   - Function: `interpret_user_intent(message, history) -> Dict[str, Any]`
4. **How it connects to the next stage:** Produces sanitized, resolved English search queries for the semantic retrieval engine in Section 13.

In [42]:
INTERPRET_PROMPT = """You are the Query Understanding module for ReCover, an insurance educational assistant for Kenyan market traders.
Your task is to analyze the user's message, understand what they want to learn, and formulate effective English search queries.

Recent Conversation History:
__HISTORY_CONTEXT__

User's Latest Message: "__MESSAGE__"

Instructions:
1. Detect Language: 'en' (English), 'sw' (Kiswahili), 'sheng' (Kenyan street code-switch/Sheng), or 'mixed'.
2. Classify Educational Intent:
   - 'definition': Asking what a term means (e.g., 'What is coverage?').
   - 'purpose': Asking why a concept exists or why they must pay (e.g., 'Why do I have to pay every month?').
   - 'process': Asking how a procedure works (e.g., 'How do I claim?').
   - 'condition_or_exclusion': Asking what is excluded or what happens if a rule is broken (e.g., 'What happens if I miss a payment?').
   - 'claims': Questions specifically about claims, documentation, or rejected payouts.
   - 'comparison': Asking the difference between two terms (e.g., 'Difference between premium and claim').
   - 'follow_up_clarification': The user is following up on the prior turn, asking for a simpler explanation, or asking 'what does that mean?'.
   - 'out_of_domain': Unrelated question (e.g., sports, politics, weather, recipes).
3. Contextual Disambiguation: If the user says 'that', 'it', or asks a short question like 'Explain more simply' or 'Why?', resolve the reference using conversation history.
4. Standalone Search Queries: Provide 1 or 2 targeted, clear English search queries aimed at retrieving the right educational facts or policy rules. Do NOT mention product recommendations unless the user specifically asked for one.

Return ONLY a valid JSON object formatted as follows:
{
  "detected_language": "en" | "sw" | "sheng" | "mixed",
  "intent": "definition" | "purpose" | "process" | "condition_or_exclusion" | "claims" | "comparison" | "follow_up_clarification" | "out_of_domain",
  "core_concept": "short description of the central topic",
  "is_educational": true | false,
  "resolved_query": "the user's question rewritten into a clear standalone sentence",
  "search_queries": ["query 1", "query 2"]
}
"""

def interpret_user_intent(message: str, history: List[Dict[str, str]]) -> Dict[str, Any]:
    """Interprets user intent, resolves follow-ups, and formulates English retrieval queries."""
    fallback = {
        "detected_language": "en",
        "intent": "definition",
        "core_concept": message,
        "is_educational": True,
        "resolved_query": message,
        "search_queries": [message]
    }

    if client is None:
        return fallback

    history_lines = []
    for turn in history[-4:]:
        role = "Trader" if turn["role"] == "user" else "ReCover"
        history_lines.append(f"{role}: {turn['text']}")
    history_context = "\n".join(history_lines) if history_lines else "(No prior conversation - this is the first message)"

    # Use direct string replacement to completely bypass Python's .format() KeyError on JSON braces
    formatted_prompt = (
        INTERPRET_PROMPT
        .replace("__HISTORY_CONTEXT__", history_context)
        .replace("__MESSAGE__", message)
    )

    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=formatted_prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type="application/json"
            )
        )
        parsed = json.loads(resp.text)
        if isinstance(parsed, dict) and "search_queries" in parsed:
            if not isinstance(parsed["search_queries"], list) or not parsed["search_queries"]:
                parsed["search_queries"] = [parsed.get("resolved_query", message)]
            return parsed
    except Exception as e:
        print(f"Notice: Intent interpretation used fallback due to error: {e}")

    return fallback

## 12. Full-Turn Conversational Memory Architecture

### Markdown Explanation
1. **What the code does:** Implements a two-sided conversation memory store (`ConversationManager`). Unlike the old notebook which threw away the assistant's previous responses and only kept raw user strings, this manager stores both user inquiries and ReCover's educational answers, timestamps, and active dialogue topics.
2. **Why that step is necessary:** Enables progressive learning: if a trader asks *"What is an excess?"*, ReCover explains it. If the trader replies *"Explain it more simply with an example from mitumba"*, the system recognizes what was previously said and pivots to a simpler analogy rather than repeating the same dictionary text.
3. **Which functions or variables it creates:**
   - Class: `ConversationManager`
   - Methods: `add_turn()`, `get_history()`, `get_context_summary()`, `clear()`.
4. **How it connects to the next stage:** Supplies multi-turn context to `interpret_user_intent()` (Section 11) and the grounded synthesizer (Section 15).

In [33]:
class ConversationManager:
    """Maintains full-turn dialogue history for both Trader and ReCover."""
    def __init__(self, max_turns: int = 6):
        self.max_turns = max_turns
        self.turns: List[Dict[str, str]] = []

    def add_user_message(self, text: str):
        self.turns.append({"role": "user", "text": text})
        self._trim()

    def add_assistant_response(self, text: str, sources: Optional[List[str]] = None):
        self.turns.append({"role": "assistant", "text": text, "sources": sources or []})
        self._trim()

    def get_history(self) -> List[Dict[str, str]]:
        return list(self.turns)

    def get_formatted_turns(self, limit: int = 4) -> str:
        """Formats recent turns for LLM prompt context."""
        recent = self.turns[-limit:] if limit else self.turns
        formatted = []
        for t in recent:
            speaker = "Trader" if t["role"] == "user" else "ReCover Educator"
            formatted.append(f"{speaker}: {t['text']}")
        return "\n\n".join(formatted) if formatted else "(No previous dialogue)"

    def clear(self):
        self.turns = []

    def _trim(self):
        # Keep only up to max_turns * 2 (user + assistant pairs)
        if len(self.turns) > self.max_turns * 2:
            self.turns = self.turns[-(self.max_turns * 2):]

print("ConversationManager class ready for multi-turn conversational memory.")

ConversationManager class ready for multi-turn conversational memory.


## 13. Semantic Retrieval and Relevance Ranking

### Markdown Explanation
1. **What the code does:** Queries the ChromaDB index using the interpreted standalone question and its rewritten variations. It deduplicates identical chunks, computes cosine similarity ($1 - distance$), filters out chunks falling below `MIN_SIMILARITY` (0.28), sorts them by relevance descending, and selects the top `MAX_RESULTS` (5 chunks).
2. **Why that step is necessary:** Vector queries can retrieve irrelevant fragments if a trader uses casual slang or metaphorical phrasing. Searching the normalized English query alongside the original text guarantees that genuine policy evidence and educational primer sections reach the LLM.
3. **Which functions or variables it creates:**
   - Function: `retrieve_evidence(interpretation: Dict[str, Any]) -> List[Dict[str, Any]]`
4. **How it connects to the next stage:** Supplies the verified text excerpts used to ground the educational response in Section 15.

In [34]:
def retrieve_evidence(interpretation: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Retrieves top relevant chunks from ChromaDB using interpreted English queries."""
    queries_to_run = interpretation.get("search_queries", [])
    resolved = interpretation.get("resolved_query", "")
    if resolved and resolved not in queries_to_run:
        queries_to_run.insert(0, resolved)

    if not queries_to_run:
        return []

    best_hits = {}

    for query_text in queries_to_run:
        # Encode query into normalized vector
        query_vec = embedder.encode(query_text, normalize_embeddings=True).tolist()

        # Query ChromaDB collection
        res = collection.query(
            query_embeddings=[query_vec],
            n_results=TOP_K,
            include=["documents", "metadatas", "distances"]
        )

        ids = res["ids"][0] if res["ids"] else []
        docs = res["documents"][0] if res["documents"] else []
        metas = res["metadatas"][0] if res["metadatas"] else []
        dists = res["distances"][0] if res["distances"] else []

        for cid, doc, meta, dist in zip(ids, docs, metas, dists):
            sim = 1.0 - dist  # Cosine similarity
            if sim >= MIN_SIMILARITY:
                if cid not in best_hits or sim > best_hits[cid]["similarity"]:
                    best_hits[cid] = {
                        "id": cid,
                        "source": meta.get("source", "Unknown"),
                        "page": meta.get("page", 0),
                        "text": doc,
                        "similarity": round(sim, 4)
                    }

    # Sort results by similarity descending
    sorted_hits = sorted(best_hits.values(), key=lambda h: h["similarity"], reverse=True)
    return sorted_hits[:MAX_RESULTS]

print("Semantic retrieval pipeline initialized with deduplication and thresholding.")

Semantic retrieval pipeline initialized with deduplication and thresholding.


## 14. Educational System Prompt & Pedagogical Guidelines

### Markdown Explanation
1. **What the code does:** Defines the master educational system instruction (`EDUCATIONAL_SYSTEM_PROMPT`) for Gemini. It instructs the model to act as an approachable, knowledgeable Kenyan insurance educator. Key operational rules:
   - Match the user's language (plain English, authentic Kiswahili, or natural Sheng).
   - Use relatable Kenyan informal trade scenarios (*mitumba* clothing bales, market kiosks, Gikomba/Toi market dynamics).
   - Distinctly separate illustrative analogies from verified contractual clauses.
   - Strictly ground all numerical rates, limits, exclusions, and statutory rules in the provided `[S#]` excerpts.
   - Say clearly when the approved documents do not contain a specific detail, without inventing rules.
   - Never push products or behave like an underwriting rejection agent.
2. **Why that step is necessary:** Replaces the old system's rigid legalistic prompt which previously forced the model to return dry statutory text or premature underwriting disqualifications.
3. **Which functions or variables it creates:**
   - String: `EDUCATIONAL_SYSTEM_PROMPT`
4. **How it connects to the next stage:** Provided as `system_instruction` during answer generation in Section 15.

In [35]:
EDUCATIONAL_SYSTEM_PROMPT = """You are 'ReCover', an approachable, patient, and knowledgeable microinsurance educator for Kenyan informal traders (e.g., mitumba clothing sellers, mama mboga, kiosk owners, market stallholders).

YOUR MISSION:
Your goal is to build genuine insurance literacy and understanding. When a trader finishes speaking with you, they should understand concepts, processes, rights, and trade-offs that they did not understand before.

TONE & LANGUAGE:
1. Grounded & Respectful: Speak like an experienced, honest mentor who understands market trade in Kenya.
2. Linguistic Mirroring: Reply in the language or mix the trader used:
   - English: Plain, clear Kenyan English (no unnecessary legal/insurance jargon).
   - Kiswahili: Warm, authentic Swahili (e.g., 'bima ya biashara', 'fidia', 'ada ya bima').
   - Sheng / Code-Switch: Natural Nairobi market phrasing (e.g., 'stock', 'duka', 'bob', 'kuchanga').
3. No Jargon: Always demystify technical terms immediately using everyday language.

PEDAGOGICAL TEACHING STRUCTURE:
- Direct Answer: Address the specific question directly in the opening 1-2 sentences.
- Concept Explanation: Explain how the concept actually works in simple terms.
- Relatable Trader Context: When helpful, illustrate using informal market scenarios (e.g., mitumba clothing bales, fire risk in open-air markets, rain damage, kiosk locks). Clearly signal that examples are for illustration.
- Conditions, Rights & Trade-offs: Point out vital conditions, limitations, or policyholder obligations so the trader gets the complete, balanced truth.
- Progressive Follow-up: Offer a natural, single-sentence conversational invitation for them to ask about related steps if they wish.

STRICT ACCURACY & DOCUMENT GROUNDING RULES:
1. Every specific policy fact, percentage rate, monetary limit, exclusion, deductible/excess, or regulatory rule must be grounded in the provided [S#] excerpts.
2. Label facts with their bracketed citation tag (e.g., [S1], [S2]).
3. NEVER INVENT: Do not invent premiums, payout promises, claim approval guarantees, or unverified insurer procedures.
4. HONEST ACKNOWLEDGMENT: If the provided excerpts do not contain the answer, say honestly and calmly that the approved documents do not specify that detail, and explain what step or insurer confirmation is typically needed.
5. NO SALES OR UNDERWRITING: Do not tell the trader they 'qualify' or 'do not qualify' for a specific product unless they explicitly asked for an eligibility audit. Do not push products. Prioritize understanding over selling.
"""

print("Educational System Prompt configured successfully.")

Educational System Prompt configured successfully.


## 15. Grounded Answer Generation & Localized Fallback

### Markdown Explanation
1. **What the code does:** Assembles the context payload (conversation history, user interpretation, and labeled excerpts `[S1]`, `[S2]`...) and calls Gemini 2.5 Flash at `temperature=0.2`. It also implements language-aware fallback messages for out-of-domain questions or cases where no relevant evidence exists.
2. **Why that step is necessary:** Ensures that answers are grounded in retrieved documents while adapting to the trader's native language. If retrieval yields nothing or the question is about football, it produces a polite refusal in English, Swahili, or Sheng.
3. **Which functions or variables it creates:**
   - Function: `generate_educational_answer(message, interpretation, evidence, history) -> Tuple[str, List[str]]`
   - Dictionary: `LOCALIZED_UNAVAILABLE`
4. **How it connects to the next stage:** Feeds raw LLM output into citation validation (Section 16).

In [36]:
LOCALIZED_UNAVAILABLE = {
    "en": "I do not have verified information about that in the approved insurance educational documents, so I cannot answer accurately without guessing. I can only explain verified microinsurance concepts, claims, premiums, and Kenyan policy guidelines.",
    "sw": "Sina maelezo yaliyothibitishwa kuhusu swali hilo kwenye nyaraka rasmi za bima, hivyo siwezi kukupa jibu la kubahatisha. Ninaweza kukusaidia kuelewa jinsi bima inavyofanya kazi, malipo ya ada (premium), madai (claims), na haki zako za kisheria nchini Kenya.",
    "sheng": "Sina hiyo info kwa ma-doc rasmi za bima, so sitaki kukupea story za jaba au kubahatisha. Naeza kukuelezea vile bima ya biashara hufanya kazi, vile unalipa premium, vile unadaisha compensation, au haki zako kama trader.",
    "mixed": "Sina verified information kuhusu hilo kwa approved documents zetu, so I cannot answer without guessing. Naweza kukusaidia kuelewa bima ya kawaida, how claims work, and what policies cover."
}

def generate_educational_answer(
    message: str,
    interpretation: Dict[str, Any],
    evidence: List[Dict[str, Any]],
    history: List[Dict[str, str]]
) -> Tuple[str, List[str]]:
    """Synthesizes a grounded educational response using retrieved excerpts and dialogue memory."""
    lang = interpretation.get("detected_language", "en")
    intent = interpretation.get("intent", "definition")

    # 1. Immediate rejection of out-of-domain queries
    if intent == "out_of_domain" or not interpretation.get("is_educational", True):
        fallback_msg = LOCALIZED_UNAVAILABLE.get(lang, LOCALIZED_UNAVAILABLE["en"])
        return fallback_msg, []

    # 2. Check for empty evidence
    if not evidence:
        fallback_msg = LOCALIZED_UNAVAILABLE.get(lang, LOCALIZED_UNAVAILABLE["en"])
        return fallback_msg, []

    # 3. Assemble labeled excerpts for Gemini
    labeled_sources = {}
    excerpt_blocks = []
    for i, chunk in enumerate(evidence, start=1):
        label = f"S{i}"
        source_label = f"{chunk['source']}, p.{chunk['page']}"
        labeled_sources[label] = source_label
        excerpt_blocks.append(f"[{label}] ({source_label}):\n{chunk['text']}")

    # Format dialogue history
    recent_history_lines = []
    for turn in history[-4:]:
        speaker = "Trader" if turn["role"] == "user" else "ReCover"
        recent_history_lines.append(f"{speaker}: {turn['text']}")
    history_str = "\n".join(recent_history_lines) if recent_history_lines else "(None - starting fresh)"

    user_payload = f"""USER QUESTION: {message}

INTERPRETATION:
- Detected Language: {lang}
- Educational Intent: {intent}
- Core Concept: {interpretation.get('core_concept', 'Insurance Education')}
- Standalone Question: {interpretation.get('resolved_query', message)}

RECENT CONVERSATION HISTORY:
{history_str}

APPROVED DOCUMENT EXCERPTS:
""" + "\n\n".join(excerpt_blocks) + """

INSTRUCTION FOR RECOVER:
Answer the trader's question in {lang} following the pedagogical instructions.
Ground your answer in the excerpts above, cite used facts with [S#], and demystify any complex terms.
"""

    if client is None:
        # Fallback simulation if no API key is present
        return f"[Offline Mode] Retrieved {len(evidence)} excerpts for topic '{interpretation.get('core_concept')}'. Configure GEMINI_API_KEY for live answers.", []

    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=user_payload,
            config=types.GenerateContentConfig(
                system_instruction=EDUCATIONAL_SYSTEM_PROMPT,
                temperature=0.2
            )
        )
        raw_text = resp.text or ""

        # Extract cited labels
        used_labels = set(re.findall(r"\[(S\d+)\]", raw_text))
        active_sources = [f"[{lbl}] {labeled_sources[lbl]}" for lbl in sorted(used_labels) if lbl in labeled_sources]

        # If model cited nothing but excerpts were relevant, provide top source
        if not active_sources and evidence:
            top = evidence[0]
            active_sources = [f"[S1] {top['source']}, p.{top['page']}"]

        return raw_text, active_sources

    except Exception as e:
        error_msg = f"An error occurred while generating the answer: {e}"
        return error_msg, []

print("Grounded answer generation module configured successfully.")

Grounded answer generation module configured successfully.


## 16. Citation Validation and Provenance Extraction

### Markdown Explanation
1. **What the code does:** Inspects the generated answer to verify that every citation tag (e.g., `[S1]`, `[S2]`) corresponds to an actual chunk passed into the model's context window. It strips out hallucinated citations and formats a clean bibliographic source list.
2. **Why that step is necessary:** Guarantees academic and audit integrity by preventing the LLM from outputting imaginary source references.
3. **Which functions or variables it creates:**
   - Function: `validate_citations(response_text: str, valid_labels: Dict[str, str]) -> Tuple[str, List[str]]`
4. **How it connects to the next stage:** Integrated directly into the `ReCoverChatbot` class in Section 17.

In [37]:
def validate_citations(response_text: str, valid_sources: Dict[str, str]) -> List[str]:
    """Extracts and validates bracketed citation tags against provided context."""
    # Find all occurrences of [S1], [S2], [S1, S2], etc.
    found_tags = []
    brackets = re.findall(r"\[([^\]]+)\]", response_text)
    for b in brackets:
        tags = re.findall(r"S\d+", b)
        for t in tags:
            if t in valid_sources and t not in found_tags:
                found_tags.append(t)

    return [f"[{t}] {valid_sources[t]}" for t in sorted(found_tags)]

print("Citation verification utility compiled.")

Citation verification utility compiled.


## 17. The Unified ReCover Chatbot Class

### Markdown Explanation
1. **What the code does:** Encapsulates the entire educational pipeline into a single, clean class: `ReCoverChatbot`. It manages multi-turn conversation memory, invokes interpretation, executes semantic retrieval, triggers grounded answer generation, and updates the conversation history.
2. **Why that step is necessary:** Eliminates fragmented functions and provides a clean, single-point interface for interactive sessions, test scripts, and evaluation benchmarks.
3. **Which functions or variables it creates:**
   - Class: `ReCoverChatbot`
   - Method: `ask(user_message: str) -> Dict[str, Any]`
   - Method: `reset()`
4. **How it connects to the next stage:** Directly powers the interactive chat function (Section 18) and evaluation suites (Sections 19–22).

In [38]:
class ReCoverChatbot:
    """Unified ReCover Educational Information Assistant."""
    def __init__(self):
        self.memory = ConversationManager(max_turns=6)

    def ask(self, user_message: str) -> Dict[str, Any]:
        """Processes a user message through the end-to-end educational pipeline."""
        user_message_clean = user_message.strip()
        if not user_message_clean:
            return {
                "answer": "Please ask an insurance question to get started.",
                "sources": [],
                "interpretation": {},
                "evidence": []
            }

        history = self.memory.get_history()

        # Step 1: Conversational Question Interpretation
        interpretation = interpret_user_intent(user_message_clean, history)

        # Step 2: Semantic Evidence Retrieval
        evidence = retrieve_evidence(interpretation)

        # Step 3: Grounded Answer Synthesis
        answer, sources = generate_educational_answer(
            message=user_message_clean,
            interpretation=interpretation,
            evidence=evidence,
            history=history
        )

        # Step 4: Update Conversation Memory
        self.memory.add_user_message(user_message_clean)
        self.memory.add_assistant_response(answer, sources)

        return {
            "answer": answer,
            "sources": sources,
            "interpretation": interpretation,
            "evidence": evidence
        }

    def reset(self):
        """Resets the conversation memory for a new session."""
        self.memory.clear()

print("ReCoverChatbot class successfully defined and ready for execution.")

ReCoverChatbot class successfully defined and ready for execution.


## 18. Interactive Chat Function

### Markdown Explanation
1. **What the code does:** Implements `chat(bot, message)` to print formatted interactions simulating a live conversation between a market trader and ReCover, clearly displaying the trader's query, ReCover's educational response, and the verified sources.
2. **Why that step is necessary:** Provides a readable, standardized demonstration interface for presentations, judges, and user testing.
3. **Which functions or variables it creates:**
   - Function: `chat(bot: ReCoverChatbot, message: str)`
4. **How it connects to the next stage:** Used to run diagnostics and test scenarios in Sections 19 through 22.

In [39]:
def chat(bot: ReCoverChatbot, message: str):
    """Convenience function to display interactive conversation turns."""
    print(f"Trader: {message}\n")
    result = bot.ask(message)
    print(f"ReCover:\n{result['answer']}\n")
    if result['sources']:
        print("Verified Sources:")
        for s in result['sources']:
            print(f"  • {s}")
    else:
        print("Sources: (None cited)")
    print("-" * 80 + "\n")

print("Interactive chat helper initialized.")

Interactive chat helper initialized.


## 19. Retrieval Diagnostics & Semantic Search Verification

### Markdown Explanation
1. **What the code does:** Inspects the semantic retrieval engine directly, bypassing the generation layer to verify that ChromaDB retrieves correct chunks for both product-specific queries and general educational inquiries.
2. **Why that step is necessary:** Verifies that the new educational primer has solved the audit finding where general literacy questions previously scored below the similarity threshold and returned empty results.
3. **Which functions or variables it creates:**
   - Test loop across educational and technical insurance terms.
4. **How it connects to the next stage:** Validates vector search quality before running the full 12-question evaluation suite in Section 20.

In [43]:
diagnostic_queries = [
    "What is a premium and why do I pay it?",
    "What happens if I miss my insurance payment?",
    "What is a deductible or policy excess?",
    "What is the premium rate for theft cover?",
    "Who will win the football match on Sunday?"  # Negative out-of-domain test
]

print("=== RUNNING RETRIEVAL DIAGNOSTICS ===\n")
for dq in diagnostic_queries:
    interp = interpret_user_intent(dq, [])
    hits = retrieve_evidence(interp)
    print(f"QUERY: '{dq}'")
    print(f"  Intent: {interp.get('intent')}, Concept: '{interp.get('core_concept')}'")
    print(f"  Search Query Formulated: {interp.get('search_queries')}")
    if hits:
        print(f"  Retrieved {len(hits)} chunks (Top match: {hits[0]['source']}, p.{hits[0]['page']} - Sim: {hits[0]['similarity']})")
    else:
        print("  -> No chunks cleared threshold (Correct behavior for out-of-domain queries)")
    print()

=== RUNNING RETRIEVAL DIAGNOSTICS ===

QUERY: 'What is a premium and why do I pay it?'
  Intent: definition, Concept: 'insurance premium definition and purpose'
  Search Query Formulated: ['What is an insurance premium and what is the purpose of paying it?', 'what is an insurance premium definition', 'why do you pay an insurance premium purpose']
  Retrieved 5 chunks (Top match: recover_microinsurance_educational_primer.txt, p.1 - Sim: 0.8204)

QUERY: 'What happens if I miss my insurance payment?'
  Intent: condition_or_exclusion, Concept: 'missed insurance premium payment and policy lapse'
  Search Query Formulated: ['What happens if a policyholder misses an insurance premium payment?', 'consequences of missed insurance premium payment grace period policy lapse', 'what happens when you miss an insurance payment in Kenya']
  Retrieved 5 chunks (Top match: recover_microinsurance_educational_primer.txt, p.2 - Sim: 0.8479)

QUERY: 'What is a deductible or policy excess?'
  Intent: definit

## 20. Realistic Question Evaluation Suite

### Markdown Explanation
1. **What the code does:** Executes ReCover against the 12 realistic trader inquiries specified in the project requirements:
   1. What is microinsurance?
   2. What is a premium?
   3. Why do I have to pay for insurance?
   4. What happens if I miss a payment?
   5. What does insurance coverage mean?
   6. What is a claim?
   7. How do I make a claim?
   8. Does insurance cover all my business losses?
   9. What happens if my claim is rejected?
   10. What should I check before buying insurance?
   11. What is the difference between a premium and a claim?
   12. How do I know what an insurance policy actually covers?
   
   It evaluates each answer, verifies citations, checks that no underwriting products were forced into the answer, and stores the results in a structured pandas DataFrame exported to `data/recover_evaluation_results.csv`.
2. **Why that step is necessary:** Provides empirical, verifiable evidence that ReCover functions as an educational assistant across all core insurance literacy topics.
3. **Which functions or variables it creates:**
   - List: `test_questions`
   - DataFrame: `df_eval`
   - File: `data/recover_evaluation_results.csv`
4. **How it connects to the next stage:** Prepares the system for multilingual testing (Section 21).

In [44]:
test_questions = [
    "What is microinsurance?",
    "What is a premium?",
    "Why do I have to pay for insurance?",
    "What happens if I miss a payment?",
    "What does insurance coverage mean?",
    "What is a claim?",
    "How do I make a claim?",
    "Does insurance cover all my business losses?",
    "What happens if my claim is rejected?",
    "What should I check before buying insurance?",
    "What is the difference between a premium and a claim?",
    "How do I know what an insurance policy actually covers?"
]

eval_results = []
print("=== EXECUTING 12-QUESTION REALISTIC EVALUATION SUITE ===\n")

for idx, question in enumerate(test_questions, start=1):
    # Fresh bot instance for independent evaluation
    bot_eval = ReCoverChatbot()
    res = bot_eval.ask(question)

    interp = res["interpretation"]
    answer_preview = res["answer"].replace("\n", " ")[:130] + "..."
    has_sources = len(res["sources"]) > 0
    sources_str = "; ".join(res["sources"]) if res["sources"] else "None"

    eval_results.append({
        "ID": f"EQ{idx:02d}",
        "Question": question,
        "Intent": interp.get("intent", "unknown"),
        "Sources Cited": sources_str,
        "Grounded": "Yes" if has_sources else "Unverified",
        "Answer Preview": answer_preview
    })

    print(f"[EQ{idx:02d}] {question}")
    print(f"  Intent: {interp.get('intent')} | Concept: {interp.get('core_concept')}")
    print(f"  Sources: {sources_str}")
    print(f"  Answer: {answer_preview}\n")

df_eval = pd.DataFrame(eval_results)
df_eval.to_csv("data/recover_evaluation_results.csv", index=False)
print("Saved evaluation metrics to data/recover_evaluation_results.csv")
display(df_eval[["ID", "Question", "Intent", "Grounded", "Sources Cited"]])

=== EXECUTING 12-QUESTION REALISTIC EVALUATION SUITE ===

[EQ01] What is microinsurance?
  Intent: definition | Concept: microinsurance definition
  Sources: [S1] THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf, p.2; [S4] THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf, p.15; [S5] ke-act-ln-2020-26-publication-document.pdf, p.1
  Answer: In simple terms, **microinsurance** is a formal, low-cost insurance product designed to protect low-income households and small bu...

[EQ02] What is a premium?
  Intent: definition | Concept: insurance premium definition
  Sources: [S2] recover_microinsurance_educational_primer.txt, p.1; [S3] ke-act-ln-2020-26-publication-document.pdf, p.4; [S5] THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf, p.2
  Answer: A **premium** is the fixed fee you pay on a regular schedule—such as daily, weekly, monthly, or yearly—to keep your insurance poli...

[EQ03] Why do I have to pay for insurance?
  Intent: purpose | Concept: purpose of paying

,ID,Question,Intent,Grounded,Sources Cited
0,EQ01,What is microinsurance?,definition,Yes,[S1] THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Fi...
1,EQ02,What is a premium?,definition,Yes,[S2] recover_microinsurance_educational_primer...
2,EQ03,Why do I have to pay for insurance?,purpose,Yes,[S1] recover_microinsurance_educational_primer...
3,EQ04,What happens if I miss a payment?,condition_or_exclusion,Yes,[S1] recover_microinsurance_educational_primer...
4,EQ05,What does insurance coverage mean?,definition,Yes,[S1] recover_microinsurance_educational_primer...
5,EQ06,What is a claim?,definition,Yes,[S1] recover_microinsurance_educational_primer...
6,EQ07,How do I make a claim?,process,Yes,[S1] recover_microinsurance_educational_primer...
7,EQ08,Does insurance cover all my business losses?,condition_or_exclusion,Yes,[S4] recover_microinsurance_educational_primer...
8,EQ09,What happens if my claim is rejected?,claims,Yes,[S1] recover_microinsurance_educational_primer...
9,EQ10,What should I check before buying insurance?,process,Yes,[S1] recover_microinsurance_educational_primer...


## 21. Multilingual Tests: Kiswahili, Sheng, and Code-Switching

### Markdown Explanation
1. **What the code does:** Tests ReCover with real Kenyan market queries in Kiswahili, Sheng, and English/Swahili code-switched phrasing. Examples:
   - Kiswahili: *"Bima ni nini na inanisaidia vipi mimi kama muuzaji wa mitumba sokoni?"*
   - Sheng: *"Nifafanulie maana ya premium na kwa nini nilazima nilipie kila mwezi hata kama sijaungukiwa na hasara?"*
   - Code-switch: *"Kama nimepata moto kwa kibanda yangu Gikomba, naanzaje kudai compensation ya stock yangu?"*
2. **Why that step is necessary:** Verifies that the question interpreter accurately detects language, formulates English search queries for vector retrieval, and instructs the synthesizer to reply in the trader's natural vernacular without translating technical jargon word-for-word.
3. **Which functions or variables it creates:**
   - Multi-turn execution across multilingual test prompts.
4. **How it connects to the next stage:** Validates language capabilities before testing conversational memory and follow-up chains in Section 22.

In [45]:
multilingual_test_cases = [
    {
        "desc": "Kiswahili Inquiry on Business Protection",
        "prompt": "Bima ni nini na inanisaidia vipi mimi kama muuzaji wa mitumba sokoni?"
    },
    {
        "desc": "Sheng Inquiry on Premiums",
        "prompt": "Nifafanulie maana ya premium na kwa nini nilazima nilipie kila mwezi hata kama sijaungukiwa na hasara?"
    },
    {
        "desc": "Code-Switched Inquiry on Fire Claims Process",
        "prompt": "Kama nimepata moto kwa kibanda yangu Gikomba, naanzaje kudai compensation ya stock yangu?"
    }
]

print("=== EXECUTING MULTILINGUAL EVALUATION ===\n")
for tc in multilingual_test_cases:
    bot_multi = ReCoverChatbot()
    print(f"TEST CASE: {tc['desc']}")
    chat(bot_multi, tc["prompt"])

=== EXECUTING MULTILINGUAL EVALUATION ===

TEST CASE: Kiswahili Inquiry on Business Protection
Trader: Bima ni nini na inanisaidia vipi mimi kama muuzaji wa mitumba sokoni?

ReCover:
Habari ndugu yangu. Kwa ufupi, **bima ya biashara ndogo (microinsurance)** ni ngao rasmi ya kifedha iliyoundwa mahsusi kuwalinda wafanyabiashara wadogo na wa sekta isiyo rasmi—kama vile wauzaji wa mitumba sokoni—dhidi ya hasara na majanga ya ghafla ya kibiashara [S2].

### Inafanya vipi kazi? (Kuchangia kwa Pamoja)
Bima inafanya kazi kupitia mfumo wa kuweka akiba ya pamoja kusaidiana kisheria (*risk pooling*) [S4]:
1. Wafanyabiashara wengi huchangia kiasi kidogo cha fedha mara kwa mara (kiasi hiki huitwa ada ya bima au **premium**) katika mfuko mmoja unaosimamiwa na kampuni ya bima iliyoidhinishwa [S4].
2. Katika mwezi fulani, wafanyabiashara wengi hawapatwi na matatizo. Hata hivyo, mfanyabiashara mmoja anapopatwa na janga lililofunikwa na bima, fedha kutoka kwenye ule mfuko wa pamoja hutumika kumlipa fidi

## 22. Conversational Continuity, Follow-Up Resolution, & Edge Cases

### Markdown Explanation
1. **What the code does:** Simulates multi-turn dialogues to test:
   - **Anaphora resolution:** Asking *"What is coverage?"* followed by *"Does that mean all goods in my stall are protected?"* (verifying that "that" is resolved to coverage).
   - **Simpler explanation request:** Asking *"Explain it more simply with an example"* (verifying that ReCover pivots to a simpler analogy rather than repeating itself).
   - **Missing information handling:** Inquiring about a policy detail not present in the documents (verifying that the model acknowledges the missing information without hallucinating).
   - **Out-of-domain handling:** Asking an unrelated question (e.g., football scores) to confirm clean rejection.
2. **Why that step is necessary:** Confirms that ReCover behaves like an attentive, adaptable teacher who retains conversation context across multiple turns.
3. **Which functions or variables it creates:**
   - Simulated conversation runs `bot_session1` and `bot_session2`.
4. **How it connects to the next stage:** Leads to Section 23, which documents current system boundaries and future expansion paths.

In [46]:
print("=== TEST SUITE 1: PROGRESSIVE EXPLANATION & ANAPHORA RESOLUTION ===\n")
bot_session1 = ReCoverChatbot()

# Turn 1: Definition
chat(bot_session1, "What does insurance coverage mean?")

# Turn 2: Follow-up relying on context ("that")
chat(bot_session1, "Does that mean everything in my shop is protected from any loss?")

# Turn 3: Request for simpler explanation with trade example
chat(bot_session1, "I am still a bit confused. Can you explain that more simply with an everyday mitumba example?")

print("\n=== TEST SUITE 2: MISSING INFORMATION & OUT-OF-DOMAIN REFUSAL ===\n")
bot_session2 = ReCoverChatbot()

# Test missing information (Specific unrecorded payout timeline)
chat(bot_session2, "Exactly how many hours does Jubilee take to pay out a claim on my specific duka?")

# Test out-of-domain rejection
chat(bot_session2, "Who will win the football match between Arsenal and Chelsea this weekend?")

=== TEST SUITE 1: PROGRESSIVE EXPLANATION & ANAPHORA RESOLUTION ===

Trader: What does insurance coverage mean?

ReCover:
In simple terms, **insurance coverage** refers to the exact scope and boundaries of protection written in your insurance contract [S1]. It clearly outlines what is protected, what specific events you are protected against, and the maximum money the insurance company will pay if a loss occurs [S1].

Think of coverage as a protective fence that has three clear parts:

1. **Insured Property:** This specifies exactly what items are protected—such as your shop’s stock, market merchandise, or business fixtures [S1].
2. **Covered Perils:** These are the specific misfortunes or dangers the insurer agrees to pay for, such as fire, lightning, explosion, or forcible burglary [S1].
3. **Policy Limits:** This is the highest payout the insurer will ever give you for a single incident or over the whole year [S1].

### A Relatable Market Example
For instance, if you operate a cloth

## 23. System Limitations, Knowledge Gaps, and Ethical Boundaries

### Technical Boundaries of the Information System
1. **Educational Scope vs. Underwriting Advisory:** ReCover is an educational assistant designed to demystify microinsurance concepts, processes, terminology, and consumer protections. It is **not** an authorized financial adviser, underwriter, or insurance broker. It does not issue binding insurance policies or guarantee claim settlements.
2. **Corpus Coverage Limitations:**
   - The knowledge base contains real Kenyan policy documents (`BRITAM-PRODUCTS-PROFILE-Final-Final.pdf`, `TCIP-BROCHURE.pdf`, `jinue_policy_document.pdf`, and the IRA Microinsurance Regulations).
   - Commercial brochures frequently quote percentage rates (e.g., `0.175%`) without specifying flat monthly premiums. ReCover correctly directs users to obtain formal quotes from licensed insurers rather than fabricating shilling estimates.
3. **Embedding Alignment:** While `all-MiniLM-L6-v2` is efficient and lightweight, it was primarily trained on English corpora. ReCover overcomes this through its LLM Question Interpretation stage, which generates normalized English search queries from Swahili and Sheng inputs before querying the vector store.
4. **Integration with the Recommendation System:** In subsequent phases of the ReCover platform, this Information System can be coupled with the teammate's Recommendation Engine via a clean API gateway. When a user transitions from general literacy inquiries (*"What is coverage?"*) to specific product matching (*"Which policy matches my KES 500/month budget?"*), the session can be handed off cleanly to the recommendation module without conflating the educational mission of this system.